In [ ]:
import pandas as pd

df_main = pd.read_csv('data/temp/neo_geo_classfied.csv')

In [33]:
df_main.info

<bound method DataFrame.info of        Unnamed: 0       doc_id  \
0              11  CHL-1946.14   
1              18  EGY-1946.18   
2              23  GBR-1946.27   
3              24  GBR-1946.28   
4              35  PHL-1946.12   
...           ...          ...   
13268       38869  VCT-2024.12   
13269       38878  VNM-2024.10   
13270       38893   ZAF-2024.3   
13271       38894  ZAF-2024.23   
13272       38895  ZAF-2024.24   

                                                    text iso_code  year  \
0      Our foreign exchange reserves are not always p...      CHL  1946   
1      Egypt is confident that the General Assembly a...      EGY  1946   
2      That will save us money — perhaps a lot; more ...      GBR  1946   
3      The first is to declare again the basic princi...      GBR  1946   
4      We trust that this General Assembly and the va...      PHL  1946   
...                                                  ...      ...   ...   
13268  It all degenerates into a M

In [34]:
df_main.columns

Index(['Unnamed: 0', 'doc_id', 'text', 'iso_code', 'year', 'final_label'], dtype='str')

In [2]:
df_main['text'].iloc[0]

'Our foreign exchange reserves are not always prepared for considerable annual disbursements by way of contributions, representation and other onerous activities. We do not sell our exportable products at sufficiently profitable prices to meet the cost of our imports in rising world markets. It is therefore desirable for the Organization to be pruned so that co-operation may be promptly effected, and the budgets should be reviewed so that proportional burdens do not become excessive. Before concluding, I should like to present, on behalf of my Government, sincere congratulations to the President of this Assembly, to the Secretary-General of the United Nations and his eight Assistant Secretaries-General for the splendid work they have performed in spite of the lack of the material facilities which the Organization needs.'

In [ ]:
#taking 50 sample from the df_main(no removing)

df_main.sample(50).to_csv('data/temp/sample_classifiled_main_50.csv', index=False)

In [ ]:
#claude code generate this function to filter the text based on keywords and save the filtered chunks in a new dataframe

import re
import pandas as pd
from nltk.tokenize import sent_tokenize

# Define your keywords
keywords = [ 'interdependence', 'flow of goods and services', 'flow of goods', 'trade exchange', 'common interests', 'market economy', 'mutual economic prosperity', 'globalization',  'liberalization', 'General Agreement on Tariffs and Trade',  'the United Nations Conference on Trade and Development', 'free movement of people', 'free movement of goods', 'free movement of goods and services', 'free market competition', 'attraction of foreign investments', 'foreign investment', 'unrestricted transit', 'unrestricted', 'GATT', 'North-South', 'foreign capital investment(s)', 'fruits', 'expansions of trade', 'World Bank', 'international financial system', 'service-based economy', 'UNCTAD', 'international economic co-operation', 'international economic cooperation', 'international cooperation', 'international collaboration', 'economic exchange', 'strategic partnerships', 'Economic and Social Council', 'trading', '*trading', 'free-market', 'free market', 'attracting investment', 'trade', 'bretton woods', 'common market', 'comparative advantage', 'cross-border trade', 'deregulation', 'duty-free', 'economic integration', 'economies of scale', 'export*', 'import', 'imports', 'importing' 'export promotion', 'export markets', 'export competitiveness', 'export-led growth', 'exporting', 'foreign demand', 'foreign trade', 'free trade agreement', 'FTA', 'global economy', 'global markets', 'global supply chains', 'globalisation', 
              'globalization', 'import competition', 'import markets', 'importing', 'Cross-border flows', 'neo-liberal*', 'neo liberal', 'neo liberalism', 'market access', 'market opening', 'opening markets', 'market-driven', 'multilateral trade', 'outsourcing', 'national treatment', 'market-led', 'interdependence', 'shared stakes', 'single market', 'supply chain*', 'trade agreement*', 
               'trade facilitation', 'trade flows', 'trade liberalisation', 'trade liberalization', 'trade volume', 'mutual benefit', 'level playing field', 'level the playing field', 'value chains', 'multilateralism', 'market-friendly', 'world market', 'world trade','WTO', 'World Trade Organization', 'Free Trade Agreement', 
               'Trade deficit(s)', 'distrust', 'confrontation', 'unequal exchange', 'trade dumping', 'unilateral*', 'non-tariff barriers', 'subsidy policies', 'embargo', 'economic warfare', 'economic boycott','asymmetry of trade', 'unjust world economic system', 'trade-distorting', 'inequality', 'blocked', 'blocks', 'unequal exchange', 'self-sufficiency', 'protectionist measures', 
                'asymetric', 'unjust', 'autarky', 'buy domestic', 'buy national', 'customs','deglobalisation', 'deglobalization', 'deficit', 'protection', 'domestic industry', 'economic nationalism', 'export subsidies', 'tariff*', 'import bans', 'import controls', 'import quotas', 'import substitution', 'quota*', 'substitution*', 'import taxes', 'industrial protection', 'infant industry', 
                'job protection', 'nationalization', 'non-tariff barriers', 'onshoring', 'protectionism', 'protectionist', 're-shoring', 'self-sufficiency', 'strategic autonomy', 'subsidised exports', 'trade barriers', 'technological supremacy', 'economic coercion', 'talent war', 'intellectual property theft', 'friend-shoring', 'unilateral', 'economic sovereignty'

]


def keyword_to_pattern(kw):
    if kw.startswith('*'):
        # wildcard at start - match any characters before the stem
        return r'\w*' + re.escape(kw[1:]) + r'\b'
    elif kw.endswith('*'):
        # wildcard at end - match any characters after the stem
        return r'\b' + re.escape(kw[:-1]) + r'\w*'
    else:
        # exact word match
        return r'\b' + re.escape(kw) + r'\b'
    

# Build pattern handling wildcards
pattern = '|'.join([keyword_to_pattern(kw) for kw in keywords])

all_chunks = []

for idx, row in df_main.iterrows():
    # Split text into sentences
    sentences = sent_tokenize(str(row['text']))
    
    # Keep only sentences that contain keywords
    filtered_sentences = [s for s in sentences if re.search(pattern, s, re.IGNORECASE)]
    
    # Skip if no sentences matched
    if not filtered_sentences:
        continue
    
    # Join filtered sentences back into a chunk
    filtered_chunk = ' '.join(filtered_sentences)
    
    # Keep all original columns and update text with filtered chunk
    new_row = row.to_dict()
    new_row['text'] = filtered_chunk
    new_row['matched_keywords'] = [kw for kw in keywords if re.search(keyword_to_pattern(kw), filtered_chunk, re.IGNORECASE)]
    new_row['num_matched_sentences'] = len(filtered_sentences)
    
    all_chunks.append(new_row)

# Create new dataframe
df_filtered = pd.DataFrame(all_chunks)

# Save
df_filtered.to_csv('data/temp/neo_geo_class_chunks.csv', index=False)

print(f"Total original chunks: {len(df_main)}")
print(f"Chunks with keywords: {len(df_filtered)}")
df_filtered.head()

Total original chunks: 13273
Chunks with keywords: 13272


,Unnamed: 0,doc_id,text,iso_code,year,final_label,matched_keywords,num_matched_sentences
0,11,CHL-1946.14,We do not sell our exportable products at suff...,CHL,1946,neo-liberal,"[export*, imports]",1
1,18,EGY-1946.18,When we consider some of the economic problems...,EGY,1946,geo-economic,"[customs, tariff*]",1
2,23,GBR-1946.27,The first is to declare again the basic princi...,GBR,1946,neo-liberal,"[interdependence, interdependence]",1
3,24,GBR-1946.28,The first is to declare again the basic princi...,GBR,1946,neo-liberal,"[interdependence, trade, interdependence]",2
4,35,PHL-1946.12,This is not the sort of economic interdependen...,PHL,1946,neo-liberal,"[interdependence, interdependence]",2


In [ ]:
#claude code generate this function to filter the text based on keywords and save the filtered chunks in a new dataframe
#we modified the code to also save the chunks that do not contain any keywords in a separate dataframe to check the problematic chunks.

all_chunks = []
no_match_chunks = []  # <--- Initialize the second list

for idx, row in df_main.iterrows():
    # Split text into sentences
    sentences = sent_tokenize(str(row['text']))
    
    # Keep only sentences that contain keywords
    filtered_sentences = [s for s in sentences if re.search(pattern, s, re.IGNORECASE)]
    
    # IF NO SENTENCES MATCHED
    if not filtered_sentences:
        # Save to the "No Match" list instead of just skipping
        no_match_chunks.append({
            'doc_id': row['doc_id'],          # Use idx or row['doc_id'] if you have a column
            'text': row['text'],
            'rethoric_label': row['final_label']
        })
        continue
    
    # IF MATCHED (Your original logic)
    filtered_chunk = ' '.join(filtered_sentences)
    new_row = row.to_dict()
    new_row['text'] = filtered_chunk
    # Check which specific keywords matched for this chunk
    new_row['matched_keywords'] = [kw for kw in keywords if re.search(keyword_to_pattern(kw), filtered_chunk, re.IGNORECASE)]
    new_row['num_matched_sentences'] = len(filtered_sentences)
    
    all_chunks.append(new_row)

# Create the original filtered dataframe
df_filtered = pd.DataFrame(all_chunks)

# Create the NEW "No Keywords" dataframe
df_no_keywords = pd.DataFrame(no_match_chunks)

# Save the new file
df_no_keywords.to_csv('data/temp/chunks_without_keywords_2.csv', index=False)

print(f"Total original chunks: {len(df_main)}")
print(f"Chunks with keywords: {len(df_filtered)}")
print(f"Chunks WITHOUT keywords: {len(df_no_keywords)}")

Total original chunks: 13273
Chunks with keywords: 13272
Chunks WITHOUT keywords: 1


In [ ]:
#identify the chunks without keywords and decided to excluded it
df_no_keywords

In [38]:
# 1. Create a new dataframe with unique text entries
df_unique = df_filtered.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)

# 2. Compare the counts to see how many were removed
original_count = len(df_filtered)
unique_count = len(df_unique)
removed_count = original_count - unique_count

print(f"Original rows: {original_count}")
print(f"Unique rows: {unique_count}")
print(f"Duplicates removed: {removed_count}")

# 3. Save to a new CSV
df_unique.to_csv('neo_geo_class_chunks.csv', index=False)

Original rows: 13272
Unique rows: 12749
Duplicates removed: 523


In [ ]:
df_unique

In [ ]:
#taking 50 examples

df_unique.sample(50).to_csv('data/temp/sample_classifiled_chunks_50.csv', index=False)

In [ ]:
#splitting the chunks into sentences

all_sentences = []

for idx, row in df_filtered.iterrows():
    # Split text into sentences
    sentences = sent_tokenize(str(row['text']))
    
    for i, sentence in enumerate(sentences, 1):
        # Create sentence id from doc_id
        new_row = row.to_dict()
        new_row['sentence_id'] = f"{row['doc_id']}_{i}"
        new_row['text'] = sentence
        all_sentences.append(new_row)

# Create new dataframe
df_sentences = pd.DataFrame(all_sentences)


print(f"Total sentences: {len(df_sentences)}")
df_sentences.head()

Total sentences: 24940


,Unnamed: 0,doc_id,text,iso_code,year,final_label,matched_keywords,num_matched_sentences,sentence_id
0,11,CHL-1946.14,We do not sell our exportable products at suff...,CHL,1946,neo-liberal,"[export*, imports]",1,CHL-1946.14_1
1,18,EGY-1946.18,When we consider some of the economic problems...,EGY,1946,geo-economic,"[customs, tariff*]",1,EGY-1946.18_1
2,23,GBR-1946.27,The first is to declare again the basic princi...,GBR,1946,neo-liberal,"[interdependence, interdependence]",1,GBR-1946.27_1
3,24,GBR-1946.28,The first is to declare again the basic princi...,GBR,1946,neo-liberal,"[interdependence, trade, interdependence]",2,GBR-1946.28_1
4,24,GBR-1946.28,"Ever since the Atlantic Charter, every Member ...",GBR,1946,neo-liberal,"[interdependence, trade, interdependence]",2,GBR-1946.28_2


In [ ]:
# 1. Create a new dataframe with unique text entries
df_unique_sen = df_sentences.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)

# 2. Compare the counts to see how many were removed
original_count = len(df_sentences)
unique_count = len(df_unique_sen)
removed_count = original_count - unique_count

print(f"Original rows: {original_count}")
print(f"Unique rows: {unique_count}")
print(f"Duplicates removed: {removed_count}")

# 3. Save to a new CSV
df_unique_sen.to_csv('data/temp/neo_geo_class_sentences.csv', index=False)

Original rows: 24940
Unique rows: 21200
Duplicates removed: 3740


In [ ]:
#taking 50 examples from the sentences df

df_unique_sen.sample(50).to_csv('data/temp/sample_classifiled_sentences_50.csv', index=False)